# Week 2: Lab Solutions - Q-Learning for Nim

---

Complete solutions to [`lab_assignment.ipynb`](lab_assignment.ipynb), in the same
order and using the same code, so you can compare cell by cell with your own
attempt. Solution code is marked with `# ---------- SOLUTION ----------`.

Looking for self-play, decaying epsilon or SARSA written in a more conventional
`gym`-style API? Those live in
[`reference_implementation.ipynb`](reference_implementation.ipynb) - but note that
notebook uses a different agent interface and does not drop into this one.

Run **Runtime -> Run all**; the whole notebook finishes in a couple of minutes.

In [ ]:
import numpy as np
import random
from collections import defaultdict
import matplotlib.pyplot as plt

print("Libraries loaded!")

---
## Part 1: The Nim Environment

Unchanged from the exercise. Note `valid_actions_for(state)`: a plain function of
the state, so we can ask about positions we are no longer standing in. The
Q-learning update in Part 3 needs exactly that.

In [ ]:
def valid_actions_for(state):
    """
    Valid actions in a Nim state, as (pile_index, num_to_remove).

    A plain function of the state, so we can also ask it about states we are
    no longer standing in - which the Q-learning update below needs.
    """
    actions = []
    for pile_idx, pile_size in enumerate(state):
        for num_remove in range(1, pile_size + 1):
            actions.append((pile_idx, num_remove))
    return actions


class NimGame:
    """
    The game of Nim.
    
    State: tuple of pile sizes, e.g., (3, 4, 5) means 3 piles with 3, 4, and 5 objects
    Action: tuple (pile_index, num_to_remove), e.g., (1, 2) means remove 2 from pile 1
    """
    
    def __init__(self, initial_piles):
        """
        Args:
            initial_piles: List or tuple of initial pile sizes, e.g., [3, 4, 5]
        """
        self.initial_piles = tuple(initial_piles)
        self.reset()
    
    def reset(self):
        """Start a new game."""
        self.piles = list(self.initial_piles)
        self.current_player = 1  # Player 1 starts
        self.done = False
        self.winner = None
        return self.get_state()
    
    def get_state(self):
        """Return current state as a tuple (hashable)."""
        return tuple(self.piles)
    
    def get_valid_actions(self):
        """
        Return list of valid actions.
        Each action is (pile_index, num_to_remove).
        """
        return valid_actions_for(self.get_state())
    
    def step(self, action):
        """
        Take an action.
        
        Args:
            action: (pile_index, num_to_remove)
            
        Returns:
            new_state, reward, done
        """
        pile_idx, num_remove = action
        
        # Validate action
        if pile_idx < 0 or pile_idx >= len(self.piles):
            raise ValueError(f"Invalid pile index: {pile_idx}")
        if num_remove < 1 or num_remove > self.piles[pile_idx]:
            raise ValueError(f"Invalid remove amount: {num_remove} from pile with {self.piles[pile_idx]}")
        
        # Make the move
        self.piles[pile_idx] -= num_remove
        
        # Check if game is over (all piles empty)
        if sum(self.piles) == 0:
            self.done = True
            # Current player took the last object, so they LOSE
            self.winner = 3 - self.current_player  # Other player wins
            reward = -1  # Taking last object is bad!
        else:
            reward = 0
        
        # Switch player
        self.current_player = 3 - self.current_player
        
        return self.get_state(), reward, self.done
    
    def render(self):
        """Display the current game state."""
        print("\nPiles:")
        for i, size in enumerate(self.piles):
            objects = '|' * size if size > 0 else '(empty)'
            print(f"  Pile {i}: {objects} ({size})")
        print(f"Player {self.current_player}'s turn")


print("NimGame class and valid_actions_for() loaded!")

### Task 1.1 / 1.2 Solution: Explore the Environment

Task 1.2 asks you to change `[5]` to `[3, 4]`. The point is to notice how the
action space is built: an action is `(pile_index, num_to_remove)`, so a two-pile
game `[3, 4]` offers `3 + 4 = 7` actions, not 2.

In [ ]:
# ---------- SOLUTION (Task 1.2): two-pile game ----------
game = NimGame([3, 4])
game.render()

print("\nValid actions:")
for action in game.get_valid_actions():
    pile_idx, num_remove = action
    print(f"  Remove {num_remove} from pile {pile_idx}")

print(f"\nTotal valid actions: {len(game.get_valid_actions())}")
print("(= sum of the pile sizes: you may take any amount from any one pile)")

---
## Part 2: A Random Opponent

Unchanged from the exercise. This is what the agent trains against.

In [ ]:
class RandomAgent:
    """An agent that picks random valid actions."""
    
    def choose_action(self, state, valid_actions):
        return random.choice(valid_actions)


# Test random agent
random_agent = RandomAgent()
game = NimGame([3, 4])

print("Random agent playing:")
state = game.reset()
while not game.done:
    game.render()
    action = random_agent.choose_action(state, game.get_valid_actions())
    print(f"  -> Removes {action[1]} from pile {action[0]}")
    state, reward, done = game.step(action)

game.render()
print(f"\nWinner: Player {game.winner}")

---
## Part 3: The Q-Learning Agent  ⭐

This is the heart of the lab. The update below is the solution to **Task 3.1**.

### Why the bandit update is not enough

In Week 1 the target *was* the reward:

```
Q(a)  <-  Q(a) + alpha * ( r - Q(a) )
```

Every pull of a bandit arm is scored immediately, so aiming at `r` is fine. Nim is
different: it pays out **only on the final move**. Aim at `r` and every move except
the last one has target 0, so the agent would learn nothing at all about its
opening - which is precisely where Nim is won or lost.

### What Q-learning changes

Q-learning aims at the reward **plus the value of where you land**:

```
Q(s,a)  <-  Q(s,a) + alpha * ( r + gamma * max_a' Q(s',a') - Q(s,a) )
```

The shape is identical - nudge the old estimate a fraction `alpha` toward a target.
Only the *target* changed. That `gamma * max_a' Q(s',a')` term is the mechanism that
drags the final win or loss backwards, one move per game played.

### What `s'` means in a two-player game

We treat the opponent as part of the environment. So from the agent's point of view,
the state *after* its action is the position it faced **on its next turn** - the
state of the next entry in its own history. The opponent's reply is just something
the environment did in between, and its randomness averages into the Q-values.

Walking the history backwards makes this easy: the move we looked at last iteration
is this move's `next_state`.

In [ ]:
class QLearningAgent:
    """
    Q-Learning agent for Nim.
    
    The Q-table maps (state, action) pairs to expected values.
    """
    
    def __init__(self, epsilon=0.1, alpha=0.5, gamma=0.9):
        """
        Args:
            epsilon: Exploration rate (probability of random action)
            alpha: Learning rate (how fast to update Q-values)
            gamma: Discount factor (importance of future rewards)
        """
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        
        # Q-table: defaultdict returns 0.0 for unseen (state, action) pairs
        self.Q = defaultdict(float)
        
        # History of moves in current game (for learning)
        self.history = []
    
    def choose_action(self, state, valid_actions):
        """
        Choose an action using epsilon-greedy strategy.
        
        With probability epsilon: choose randomly (explore)
        Otherwise: choose the action with highest Q-value (exploit)
        """
        # Exploration: random action
        if random.random() < self.epsilon:
            return random.choice(valid_actions)
        
        # Exploitation: best action based on Q-values
        q_values = [self.Q[(state, action)] for action in valid_actions]
        max_q = max(q_values)
        
        # If multiple actions have the same Q-value, pick randomly among them
        best_actions = [a for a, q in zip(valid_actions, q_values) if q == max_q]
        return random.choice(best_actions)
    
    def store_transition(self, state, action):
        """Remember this move for learning later."""
        self.history.append((state, action))
    
    def learn(self, final_reward):
        """
        Update Q-values based on the game outcome.

        We work backwards through the history, from the last move to the first.
        Nim only pays out at the end, so every move except the last one gets a
        reward of 0 and has to learn purely from what came after it.

        From the agent's point of view the opponent is part of the environment,
        so the "next state" for a move is simply the position the agent faced
        when it next got a turn: the state of the next entry in its own history.

        Args:
            final_reward: 1 if won, -1 if lost, 0 for draw
        """
        # The state the agent faced on its *next* move. None = the game ended,
        # so there is no future to bootstrap from.
        next_state = None

        # Process moves in reverse order (from game end to start)
        for state, action in reversed(self.history):
            # Current Q-value
            current_q = self.Q[(state, action)]

            # ---------- SOLUTION (Task 3.1) ----------
            if next_state is None:
                # Terminal move: the game's verdict is the whole target.
                target = final_reward
            else:
                # Non-terminal move: Nim gives r = 0 here, so the target is just
                # the discounted value of the best move available next.
                next_q_values = [self.Q[(next_state, a)] for a in valid_actions_for(next_state)]
                target = self.gamma * max(next_q_values)

            self.Q[(state, action)] = current_q + self.alpha * (target - current_q)
            # -----------------------------------------
            # Walking backwards: this move's state is the "next state" of the
            # move that preceded it.
            next_state = state

        # Clear history for next game
        self.history = []


print("QLearningAgent (solution) loaded!")

---
## Part 4: Training

In [ ]:
def train_agent(agent, opponent, game, n_games=1000, agent_player=1):
    """
    Train the Q-learning agent by playing games.
    
    Args:
        agent: The Q-learning agent to train
        opponent: The opponent agent (e.g., RandomAgent)
        game: The Nim game environment
        n_games: Number of games to play
        agent_player: Which player the agent is (1 or 2)
    
    Returns:
        wins, losses: Number of wins and losses
    """
    wins = 0
    losses = 0
    
    for _ in range(n_games):
        state = game.reset()
        
        while not game.done:
            valid_actions = game.get_valid_actions()
            
            if game.current_player == agent_player:
                # Agent's turn
                action = agent.choose_action(state, valid_actions)
                agent.store_transition(state, action)
            else:
                # Opponent's turn
                action = opponent.choose_action(state, valid_actions)
            
            state, reward, done = game.step(action)
        
        # Game over - learn from result
        if game.winner == agent_player:
            agent.learn(1.0)  # Win!
            wins += 1
        else:
            agent.learn(-1.0)  # Loss
            losses += 1
    
    return wins, losses


print("Training function loaded!")

### Task 4.1 Solution: Train on a Simple Game (1 Pile)

In [ ]:
# ---------- SOLUTION (Task 4.1) ----------
random.seed(0)  # so the numbers below are reproducible

agent = QLearningAgent(epsilon=0.3, alpha=0.5, gamma=0.9)
opponent = RandomAgent()
game = NimGame([5])  # Single pile of 5

print("Training on 1 pile of 5 objects...")
wins, losses = train_agent(agent, opponent, game, n_games=5000, agent_player=1)

print(f"\nTraining complete!")
print(f"Wins: {wins} ({wins/50:.1f}%)")
print(f"Losses: {losses} ({losses/50:.1f}%)")
print(f"Q-table entries: {len(agent.Q)}")
print("\nNote: this win rate includes exploration (epsilon=0.3), so it understates")
print("how good the agent actually is. Part 7 evaluates it greedily.")

### Task 4.2 Solution: Inspect What the Agent Learned

In [ ]:
def show_q_values(agent, state, valid_actions):
    """Display Q-values for all valid actions in a state."""
    print(f"\nState: {state}")
    print("Q-values:")
    
    q_values = [(action, agent.Q[(state, action)]) for action in valid_actions]
    q_values.sort(key=lambda x: x[1], reverse=True)  # Sort by Q-value
    
    for action, q in q_values:
        pile_idx, num_remove = action
        marker = " <-- BEST" if q == max(v[1] for v in q_values) else ""
        print(f"  Remove {num_remove} from pile {pile_idx}: Q = {q:+.3f}{marker}")


# Show Q-values for starting state
game = NimGame([5])
state = game.get_state()
show_q_values(agent, state, game.get_valid_actions())

### Answer: What's the Best Opening Move?

**Remove 4 from pile 0**, leaving a single object. The Q-values make the whole
strategy readable:

| Move | Leaves | Q-value | Why |
|---|---|---|---|
| Remove 4 | `(1,)` | **+1.000** | Opponent *must* take the last object and lose. A forced win. |
| Remove 2 | `(3,)` | ~ +0.2 | Playable, but the opponent can still take 2 and hand the loss back. |
| Remove 3 | `(2,)` | ~ -0.1 | Opponent takes 1, leaving you with the last object. |
| Remove 1 | `(4,)` | ~ -0.4 | Opponent can take 3 and leave you with one. |
| Remove 5 | `()` | **-1.000** | You take the last object. Immediate loss. |

Two things worth pointing out to yourself:

- The two *certain* outcomes converged to exactly **+1.000 and -1.000**. Those come
  straight from `final_reward` with no discounting, because they are terminal or
  one move from terminal.
- The middle moves sit strictly between. They are not certainties - they depend on
  what a *random* opponent does next, and the Q-values have averaged over that.
  Against a perfect opponent they would all be -1.

---
## Part 5: Play Against Your Agent

Unchanged from the exercise. Uncomment the call to play.

In [ ]:
def play_against_agent(agent, initial_piles, human_player=1):
    """
    Play interactively against the trained agent.
    
    Args:
        agent: The trained Q-learning agent
        initial_piles: Starting pile configuration
        human_player: 1 to go first, 2 to go second
    """
    game = NimGame(initial_piles)
    agent.epsilon = 0  # No exploration during play
    
    print("\n" + "="*50)
    print("NIM vs Q-Learning Agent")
    print("="*50)
    print(f"You are Player {human_player} ({'first' if human_player == 1 else 'second'})")
    print("Remember: Taking the LAST object means you LOSE!")
    print("\nHow to play:")
    print("  Enter: pile_number, amount_to_remove")
    print("  Example: 0, 3 (removes 3 from pile 0)")
    
    state = game.reset()
    
    while not game.done:
        game.render()
        valid_actions = game.get_valid_actions()
        
        if game.current_player == human_player:
            # Human's turn
            while True:
                try:
                    user_input = input("Your move (pile, amount): ")
                    pile_idx, num_remove = map(int, user_input.replace(' ', '').split(','))
                    action = (pile_idx, num_remove)
                    if action in valid_actions:
                        break
                    print(f"Invalid move! Valid actions: {valid_actions}")
                except (ValueError, IndexError):
                    print("Please enter: pile_number, amount (e.g., 0, 2)")
        else:
            # Agent's turn
            action = agent.choose_action(state, valid_actions)
            print(f"Agent removes {action[1]} from pile {action[0]}")
        
        state, _, _ = game.step(action)
    
    # Game over
    game.render()
    if game.winner == human_player:
        print("\n🎉 Congratulations! You won!")
    else:
        print("\n🤖 The agent wins!")


print("Play function loaded!")
print("\nTo play, run: play_against_agent(agent, [5], human_player=1)")

---
## Part 6 Solutions: Harder Configurations

The pattern is the same each time; only the pile configuration and the number of
training games change. Bigger state spaces need more games, and the growth is steep
(see Question 3).

In [ ]:
# ---------- SOLUTION (Task 6.1): two piles ----------
agent_2pile = QLearningAgent(epsilon=0.3, alpha=0.5, gamma=0.9)
game_2pile = NimGame([3, 4])

print("Training on 2 piles [3, 4]...")
wins, losses = train_agent(agent_2pile, opponent, game_2pile, n_games=10000, agent_player=1)
print(f"Q-table entries: {len(agent_2pile.Q)}\n")

# ---------- SOLUTION (Task 6.2): three piles ----------
agent_3pile = QLearningAgent(epsilon=0.3, alpha=0.5, gamma=0.9)
game_3pile = NimGame([2, 3, 4])

print("Training on 3 piles [2, 3, 4]...")
wins, losses = train_agent(agent_3pile, opponent, game_3pile, n_games=20000, agent_player=1)
print(f"Q-table entries: {len(agent_3pile.Q)}\n")

# ---------- SOLUTION (Task 6.3): four piles ----------
agent_4pile = QLearningAgent(epsilon=0.3, alpha=0.5, gamma=0.9)
game_4pile = NimGame([1, 3, 5, 7])

print("Training on 4 piles [1, 3, 5, 7]...")
wins, losses = train_agent(agent_4pile, opponent, game_4pile, n_games=50000, agent_player=1)
print(f"Q-table entries: {len(agent_4pile.Q)}")

---
## Part 7: Evaluation (no exploration)

In [ ]:
def evaluate_agent(agent, game, n_games=1000):
    """Evaluate agent win rate (no exploration)."""
    original_epsilon = agent.epsilon
    agent.epsilon = 0
    
    wins_as_p1 = 0
    wins_as_p2 = 0
    opponent = RandomAgent()
    
    # Test as Player 1
    for _ in range(n_games // 2):
        state = game.reset()
        while not game.done:
            valid_actions = game.get_valid_actions()
            if game.current_player == 1:
                action = agent.choose_action(state, valid_actions)
            else:
                action = opponent.choose_action(state, valid_actions)
            state, _, _ = game.step(action)
        if game.winner == 1:
            wins_as_p1 += 1
    
    # Test as Player 2
    for _ in range(n_games // 2):
        state = game.reset()
        while not game.done:
            valid_actions = game.get_valid_actions()
            if game.current_player == 2:
                action = agent.choose_action(state, valid_actions)
            else:
                action = opponent.choose_action(state, valid_actions)
            state, _, _ = game.step(action)
        if game.winner == 2:
            wins_as_p2 += 1
    
    agent.epsilon = original_epsilon
    return wins_as_p1 / (n_games // 2), wins_as_p2 / (n_games // 2)


# Evaluate all agents
print("Agent Performance vs Random Opponent:\n")

configs = [
    ("1 pile [5]", agent, NimGame([5])),
    ("2 piles [3,4]", agent_2pile, NimGame([3, 4])),
    ("3 piles [2,3,4]", agent_3pile, NimGame([2, 3, 4])),
    ("4 piles [1,3,5,7]", agent_4pile, NimGame([1, 3, 5, 7])),
]

for name, ag, gm in configs:
    win_p1, win_p2 = evaluate_agent(ag, gm, n_games=1000)
    print(f"{name:20s} | As P1: {win_p1*100:5.1f}% | As P2: {win_p2*100:5.1f}%")

### What you should see

Greedy evaluation, 2000 games per configuration, agents trained **as Player 1**:

| Config | As P1 | As P2 | Q entries |
|---|---|---|---|
| 1 pile `[5]` | 100% | ~73% | 11 |
| 2 piles `[3,4]` | 100% | ~92% | 58 |
| 3 piles `[2,3,4]` | 100% | ~96% | 240 |
| 4 piles `[1,3,5,7]` | ~92% | ~98% | 2948 |

Your numbers will differ by a few points from run to run - and more if you changed
the seed, `epsilon` or the number of training games.

---
## Part 8: Answers to the Analysis Questions

### Question 1: Why does the agent perform better in certain player positions?

**Because it only ever trained in one of them.** Every agent above learned as
Player 1, with `agent_player=1`. As Player 1 it moves on even-numbered positions;
as Player 2 it faces the odd ones. Those are largely *different states*, and the
ones it meets as P2 are under-trained or absent from the Q-table - a
`defaultdict(float)` miss returns 0.0, so `choose_action` is effectively guessing
among ties.

This is most visible in 1 pile `[5]`: ~100% as P1 collapsing to ~74% as P2. With
one pile the game is short, so P1 and P2 see almost disjoint positions and there is
nothing to transfer.

There is a second, independent effect on top: **some positions are simply lost on
principle.** `[1,3,5,7]` is the famous losing opening for the player to move (see
Q2), which is why it is the one configuration where the agent does *better* as P2
(~98%) than as P1 (~95%).

> **Fix it yourself:** train with `agent_player` alternating randomly between 1 and
> 2, or train by self-play (Bonus 1). Either makes the agent position-agnostic.

### Question 2: What happens in `[1,3,5,7]` if both players play perfectly?

**Player 2 wins.** The Nim-sum (XOR of pile sizes) is

```
1 XOR 3 XOR 5 XOR 7 = 0
```

A position with Nim-sum 0 is a **P-position**: a loss for the player about to move.
Any move you make from a zero Nim-sum necessarily makes it non-zero, and your
opponent can always move it back to zero. Repeat, and you are the one handed the
last object.

So against a perfect opponent, perfect play from Player 1 still loses - the agent's
~92% as P1 is purely because `RandomAgent` is nowhere near perfect and keeps handing
the position back. Note what this means: **a high win rate against a random opponent
is not evidence of good play.** The agent is exploiting a weak opponent, and it never
had the chance to learn otherwise, because it never faced anything better. (This is
exactly the gap self-play closes.)

*(Caveat on the strategy stated in the week README: the plain XOR rule is for
**normal-play** Nim, where taking the last object* wins*. This lab is* misère *Nim -
taking the last object* loses*. The optimal strategy is the same XOR rule until all
remaining piles have size 1, at which point it inverts. For `[1,3,5,7]` the
zero-Nim-sum verdict still stands.)*

### Question 3: Why does the Q-table grow so fast with more piles?

Because the number of reachable positions is **multiplicative in the piles, not
additive.** A pile of size *n* can be in any of *n+1* states (`0..n`), independently
of the others, so a configuration `[a, b, c, d]` has up to

```
(a+1)(b+1)(c+1)(d+1)
```

positions - and the Q-table stores one entry per *(state, action)* pair, which
multiplies it again by the number of legal moves.

| Config | Reachable states | Q entries observed |
|---|---|---|
| `[5]` | 6 | 11 |
| `[3,4]` | 20 | 58 |
| `[2,3,4]` | 60 | 240 |
| `[1,3,5,7]` | 384 | 2948 |

Adding one pile multiplies the state space; it does not extend it. And since every
entry has to be *visited* several times before its value is reliable, the training
games needed grow with it.

**This is the wall that motivates the rest of the course.** A table cannot generalise
- learning `(1,3,5,7)` teaches it nothing about `(1,3,5,6)`, even though the two are
near-identical. Replace the table with a function approximator (a neural network)
and similar states get similar values for free. That is the whole idea behind deep
RL.

---
## Bonus 1 Solution: Self-Play

The agent plays both sides. The trick is to keep a separate history per side and
hand each one its own verdict - so we keep two agents sharing nothing but the game.

In [ ]:
# ---------- SOLUTION (Bonus 1) ----------
def train_self_play(game, n_games=20000, epsilon=0.3, alpha=0.5, gamma=0.9):
    """
    Train two Q-learning agents against each other.

    Each agent stores only its own moves, and learns from its own result, so the
    existing learn() works unchanged. Returns both agents.
    """
    agents = {1: QLearningAgent(epsilon, alpha, gamma),
              2: QLearningAgent(epsilon, alpha, gamma)}

    for _ in range(n_games):
        state = game.reset()

        while not game.done:
            player = game.current_player
            action = agents[player].choose_action(state, game.get_valid_actions())
            agents[player].store_transition(state, action)
            state, _, _ = game.step(action)

        # Each side learns from its own outcome.
        for player, ag in agents.items():
            ag.learn(1.0 if game.winner == player else -1.0)

    return agents[1], agents[2]


random.seed(1)
sp_p1, sp_p2 = train_self_play(NimGame([3, 4]), n_games=20000)

print("Self-play agents vs RandomAgent (greedy):")
for label, ag in [("trained as P1", sp_p1), ("trained as P2", sp_p2)]:
    w1, w2 = evaluate_agent(ag, NimGame([3, 4]), n_games=2000)
    print(f"  {label}: as P1 {w1*100:5.1f}% | as P2 {w2*100:5.1f}%  (Q entries {len(ag.Q)})")

print("\nCompare with the vs-random agent from Part 7, which is strong as P1 but")
print("noticeably weaker as P2. A self-play agent meets both sides of the board.")

---
## Bonus 2 Solution: Learning Curve

Train in chunks, evaluating greedily between them.

In [ ]:
# ---------- SOLUTION (Bonus 2) ----------
def train_and_track(game, n_games=50000, eval_every=2000, n_eval=400):
    """
    Train an agent and record its greedy win rate along the way.

    Returns (game_numbers, win_rates) - win rate measured as Player 1.
    """
    agent = QLearningAgent(epsilon=0.3, alpha=0.5, gamma=0.9)
    opponent = RandomAgent()

    game_numbers, win_rates = [0], []
    win_rates.append(evaluate_agent(agent, game, n_games=n_eval)[0])

    for start in range(0, n_games, eval_every):
        train_agent(agent, opponent, game, n_games=eval_every, agent_player=1)
        game_numbers.append(start + eval_every)
        win_rates.append(evaluate_agent(agent, game, n_games=n_eval)[0])

    return agent, game_numbers, win_rates


random.seed(2)
tracked_agent, game_nums, win_rates = train_and_track(NimGame([2, 3, 4]), n_games=40000)

plt.figure(figsize=(8, 4.5))
plt.plot(game_nums, [w * 100 for w in win_rates], marker='o', markersize=3)
plt.axhline(50, ls='--', lw=1, color='grey', label='coin flip')
plt.xlabel('Training games')
plt.ylabel('Greedy win rate as P1 (%)')
plt.title('Learning curve - Nim [2, 3, 4] vs random opponent')
plt.ylim(0, 105)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("The curve rises fast and then flattens: the opening moves are learned first")
print("(they are visited every single game), the rare deep positions last.")

---
## Bonus 3 Solution: Hyperparameters

A small sweep on `[2, 3, 4]`. Each cell is a greedy win rate as Player 1 after
20,000 training games.

In [ ]:
# ---------- SOLUTION (Bonus 3) ----------
def quick_train(piles, n_games=20000, **kwargs):
    agent = QLearningAgent(**kwargs)
    train_agent(agent, RandomAgent(), NimGame(piles), n_games=n_games, agent_player=1)
    return evaluate_agent(agent, NimGame(piles), n_games=1000)[0]


random.seed(3)

print("Effect of epsilon (alpha=0.5, gamma=0.9):")
for eps in [0.05, 0.1, 0.3, 0.5, 1.0]:
    wr = quick_train([2, 3, 4], epsilon=eps, alpha=0.5, gamma=0.9)
    print(f"  epsilon={eps:<5} -> {wr*100:5.1f}%")

print("\nEffect of alpha (epsilon=0.3, gamma=0.9):")
for al in [0.05, 0.1, 0.5, 0.9, 1.0]:
    wr = quick_train([2, 3, 4], epsilon=0.3, alpha=al, gamma=0.9)
    print(f"  alpha={al:<5}   -> {wr*100:5.1f}%")

print("\nEffect of gamma (epsilon=0.3, alpha=0.5):")
for ga in [0.5, 0.9, 0.99, 1.0]:
    wr = quick_train([2, 3, 4], epsilon=0.3, alpha=0.5, gamma=ga)
    print(f"  gamma={ga:<5}   -> {wr*100:5.1f}%")

### What the sweep shows

Read these as tendencies, not precise measurements - a single 1000-game evaluation
carries a couple of points of noise, so neighbouring rows can swap places.

- **epsilon** - the effect is non-monotonic, worst at both ends. Too low (`0.05`)
  and whole branches of the game tree are never visited, so their Q-values sit at
  the `defaultdict` default of 0.0 and the agent guesses there. Too high and moves
  are spent on positions good play never reaches. Note that `epsilon=1.0` - a purely
  random player - still learns a decent policy: the update is **off-policy**, so
  `max_a' Q(s',a')` evaluates the greedy policy no matter how the moves were
  actually chosen. Exploration affects *which* states get learned, not *what* is
  learned about them.
- **alpha** - high `alpha` clearly hurts (`1.0` is the worst setting in the sweep).
  It is tempting to think Nim is deterministic so a big step is safe, but the
  *target* is not deterministic: `max_a' Q(s',a')` is taken at the position left
  after a **random** opponent's reply, so it varies from game to game. `alpha=1.0`
  throws away the old estimate and adopts whatever the last sample said, so nothing
  ever averages. Small `alpha` is merely slow.
- **gamma** - matters more here than you might expect, because the *only* reward is
  terminal. With `gamma=0.5`, the value reaching a move four plies from the end is
  scaled by `0.5^4` ~ 0.06, so early moves get a very faint signal and the opening
  is learned poorly. Values near 1 do best. The discount would matter less if
  rewards were spread through the episode.

The honest summary: on a problem this small, most reasonable settings get close, and
the two parameters that bite - `alpha` and `gamma` - bite for reasons specific to a
sparse, terminal-only reward.

---
## Summary

1. ✅ Q-learning applied to a turn-based game, with the opponent folded into the
   environment
2. ✅ The update rule implemented: same shape as the bandit update, new target
3. ✅ Agents trained on progressively harder configurations
4. ✅ Interactive play against the trained agents
5. ✅ Analysed how the state space - and the training cost - grows

**Three things worth carrying forward:**

- The bandit update and the Q-learning update differ *only in what they aim at*.
  Everything else in this notebook was bookkeeping.
- **A win rate against a weak opponent measures the opponent, not the agent.**
  `[1,3,5,7]` is theoretically lost for Player 1, and our agent still "wins" 95% of
  the time.
- A Q-table cannot generalise between similar states, and the number of states
  multiplies with every pile added. That wall is what function approximation exists
  to break.